<a href="https://colab.research.google.com/github/JillianneKateLBocol/Final-Project-CS2/blob/main/Kamia_Ecommerce_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import json
with open("/content/Ecommerce.JSON") as f:
  orders = json.load(f)
print(orders)

[{'order_id': 1, 'customer': 'Alice Brown', 'order_date': '2025-08-01', 'status': 'Delivered', 'payment_method': 'Credit Card', 'shipping_address': '123 Main St, Quezon City', 'items': [{'product': 'Headphones', 'category': 'Electronics', 'price': 1500, 'quantity': 1}, {'product': 'USB-C Cable', 'category': 'Accessories', 'price': 250, 'quantity': 2}], 'total_amount': 2000}, {'order_id': 2, 'customer': 'Bob White', 'order_date': '2025-08-03', 'status': 'Pending', 'payment_method': 'Cash on Delivery', 'shipping_address': '45 Mabini St, Davao City', 'items': [{'product': 'Smartphone', 'category': 'Electronics', 'price': 22000, 'quantity': 1}, {'product': 'Phone Case', 'category': 'Accessories', 'price': 500, 'quantity': 1}], 'total_amount': 22500}, {'order_id': 3, 'customer': 'Charlie Green', 'order_date': '2025-08-05', 'status': 'Delivered', 'payment_method': 'GCash', 'shipping_address': '78 Rizal Ave, Cebu City', 'items': [{'product': 'Book', 'category': 'Books', 'price': 450, 'quantit

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install firebase-admin

In [6]:
import firebase_admin
from firebase_admin import credentials

path_to_key = "/content/Firebase.JSON"

if not firebase_admin._apps:
    cred = credentials.Certificate(path_to_key)
    firebase_admin.initialize_app(cred, {
        'databaseURL': "https://kamia-ecommerce-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase connected successfully!")
else:
    print("Firebase is already connected.")

Firebase connected successfully!


In [7]:
from firebase_admin import db

ref = db.reference("orders")

for order in orders:
    ref.child(str(order["order_id"])).set(order)

print("Data uploaded successfully!")

Data uploaded successfully!


In [8]:
ref = db.reference("orders")
orders_data = ref.get()

In [9]:
while True:
    print("\n===== ECOMMERCE DATABASE MENU =====")
    print("1. Display All Orders")
    print("2. Add New Order")
    print("3. Update Order")
    print("4. Delete Order")
    print("5. Features")
    print("6. Exit")

    choice = input("Enter choice: ")

    # 1. Display
    if choice == "1":
      orders_data = ref.get()
      if orders_data:
        print("\n" + "="*50)
        print("          DETAILED ORDER HISTORY")
        print("="*50)

        if isinstance(orders_data, list):
            items_to_show = [(i, data) for i, data in enumerate(orders_data) if data]
        else:
            items_to_show = orders_data.items()

        for oid, data in items_to_show:
            print(f"\n[ ORDER ID: {oid} ]")
            print(f"Customer: {data.get('customer', 'N/A')}")
            print(f"Date:     {data.get('order_date', 'N/A')}")
            print(f"Status:   {data.get('status', 'N/A')}")
            print(f"Payment:  {data.get('payment_method', 'N/A')}")
            print(f"Address:  {data.get('shipping_address', 'N/A')}")

            print("-" * 30)
            print("ITEMS PURCHASED:")

            current_items = data.get('items', [])

            if not current_items:
                print("  (No items listed)")
            else:
                for i, item in enumerate(current_items, 1):
                    p_name = item.get('product', 'Unknown')
                    p_qty  = item.get('quantity', 0)
                    p_price = item.get('price', 0)
                    p_cat   = item.get('category', 'N/A')
                    print(f"  {i}. {p_name} [{p_cat}]")
                    print(f"     Qty: {p_qty} | Price: ₱{p_price} | Subtotal: ₱{p_qty * p_price}")

            print("-" * 30)
            print(f"GRAND TOTAL: ₱{data.get('total_amount', 0)}")
            print("="*50)
      else:
        print("\nNo orders found in the database.")

    # 2. Add
    elif choice == "2":
      oid = input("Enter Order ID: ")
      customer = input("Enter Customer Name: ")
      order_date = input("Enter Order Date (YYYY-MM-DD): ")
      payment_method = input("Enter Payment Method: ")
      shipping_address = input("Enter Shipping Address: ")
      status = "Pending"

      items = []
      while True:
        prod_name = input("Enter Product Name (or 'done' to finish): ")
        if prod_name.lower() == 'done':
          break

          category = input("Enter Category: ")
          price = float(input("Enter Price: "))
          quantity = int(input("Enter Quantity: "))

          items.append({
            "product": prod_name,
            "category": category,
            "price": price,
            "quantity": quantity
          })

      total_amount = sum(item['price'] * item['quantity'] for item in items)

      order_data = {
        "order_id": int(oid),
        "customer": customer,
        "order_date": order_date,
        "status": status,
        "payment_method": payment_method,
        "shipping_address": shipping_address,
        "items": items,
        "total_amount": total_amount
      }

      ref.child(oid).set(order_data)
      print(f"\nOrder #{oid} for {customer} created successfully! Total: ₱{total_amount}")

    # 3. Update
    elif choice == "3":
      oid = input("Enter Order ID to update: ")
      order_ref = ref.child(oid)
      order_data = order_ref.get()

      if order_data:
        print(f"\n--- Updating Order #{oid} (Press Enter to keep current value) ---")

        # Update basic info
        new_customer = input(f"Customer [{order_data.get('customer', 'N/A')}]: ") or order_data.get('customer', 'N/A')
        new_date = input(f"Date [{order_data.get('order_date', 'N/A')}]: ") or order_data.get('order_date', 'N/A')
        new_status = input(f"Status [{order_data.get('status', 'Pending')}]: ") or order_data.get('status', 'Pending')
        new_payment = input(f"Payment [{order_data.get('payment_method', 'N/A')}]: ") or order_data.get('payment_method', 'N/A')
        new_address = input(f"Address [{order_data.get('shipping_address', 'N/A')}]: ") or order_data.get('shipping_address', 'N/A')

        # FIX: Use .get('items', []) so it doesn't crash if 'items' is missing
        current_items = order_data.get('items', [])
        new_items = []

        if not current_items:
            print("\n(No items currently in this order.)")
        else:
            print("\n--- Current Items ---")
            for i, item in enumerate(current_items):
                # Another safety check for individual item fields
                old_prod = item.get('product', 'Unknown')
                print(f"Item {i}: {old_prod}")

                u_prod = input(f"  Product Name [{old_prod}]: ") or old_prod
                u_cat = input(f"  Category [{item.get('category', 'N/A')}]: ") or item.get('category', 'N/A')
                u_price = input(f"  Price [{item.get('price', 0)}]: ")
                u_qty = input(f"  Quantity [{item.get('quantity', 0)}]: ")

                u_price = float(u_price) if u_price else item.get('price', 0)
                u_qty = int(u_qty) if u_qty else item.get('quantity', 0)

                new_items.append({
                    "product": u_prod,
                    "category": u_cat,
                    "price": u_price,
                    "quantity": u_qty
                })

        # Recalculate total
        new_total = sum(it['price'] * it['quantity'] for it in new_items) if new_items else order_data.get('total_amount', 0)

        # Apply updates
        updated_data = {
            "customer": new_customer,
            "order_date": new_date,
            "status": new_status,
            "payment_method": new_payment,
            "shipping_address": new_address,
            "items": new_items,
            "total_amount": new_total
        }

        order_ref.update(updated_data)
        print(f"\nOrder #{oid} updated successfully!")
      else:
        print("Order ID not found.")

    # 4. Delete
    elif choice == "4":
        oid = input("Enter Order ID to delete: ")

        if ref.child(oid).get():
            ref.child(oid).delete()
            print(f"Order #{oid} has been deleted.")
        else:
            print("Order ID not found.")

    # 5. Features
    elif choice == "5":
        while True:
            print("\n===== ECOMMERCE DATABSE FEATURES =====")
            print("1. Show total orders and total revenue")
            print("2. Show best-selling product")
            print("3. Show top 3 customers")
            print("4. Show number of orders by status")
            print("5. Search sales of a product")
            print("6. Back to main menu")
            choice_feature = input("Enter your choice: ")

            orders_data = ref.get()

            if not orders_data:
                print("No data available.")
                break

            orders = list(orders_data.values()) if isinstance(orders_data, dict) else orders_data

            match choice_feature:
                case "1":
                    total_orders = len([o for o in orders if o])
                    total_revenue = sum(order["total_amount"] for order in orders if order)
                    print(f"\nTotal Orders: {total_orders}")
                    print(f"Total Revenue: ₱{total_revenue}")

                case "2":
                    product_sales = {}
                    for order in orders:
                        if order and "items" in order:
                            for item in order["items"]:
                                p = item["product"]
                                product_sales[p] = product_sales.get(p, 0) + item["quantity"]

                    if product_sales:
                        best_product = max(product_sales, key=product_sales.get)
                        print(f"\nBest-Selling Product: {best_product}")
                        print(f"Total Quantity Sold: {product_sales[best_product]}")

                case "3":
                    customer_spending = {}
                    for order in orders:
                        if order:
                            c = order["customer"]
                            customer_spending[c] = customer_spending.get(c, 0) + order["total_amount"]

                    top_customers = sorted(customer_spending.items(), key=lambda x: x[1], reverse=True)[:3]
                    print("\nTop 3 Customers:")
                    for i, (customer, amount) in enumerate(top_customers, 1):
                        print(f"{i}. {customer} - ₱{amount}")

                case "4":
                    status_count = {"Pending": 0, "Shipped": 0, "Delivered": 0}
                    for order in orders:
                        if order and order["status"] in status_count:
                            status_count[order["status"]] += 1

                    print("\nOrder Status Count:")
                    for status, count in status_count.items():
                        print(f"{status}: {count}")

                case "5":
                    search_product = input("Enter product name: ").lower()
                    total_qty = 0
                    for order in orders:
                        if order and "items" in order:
                            for item in order["items"]:
                                if item["product"].lower() == search_product:
                                    total_qty += item["quantity"]
                    print(f"\nTotal '{search_product}' sold: {total_qty}")

                case "6":
                    break

                case _:
                    print("Invalid choice.")

    # 6. Exit
    elif choice == "6":
        print("Exiting System...")
        break

    else:
        print("Invalid choice. Please try again.")


===== ECOMMERCE DATABASE MENU =====
1. Display All Orders
2. Add New Order
3. Update Order
4. Delete Order
5. Features
6. Exit
Enter choice: 1

          DETAILED ORDER HISTORY

[ ORDER ID: 1 ]
Customer: Alice Brown
Date:     2025-08-01
Status:   Delivered
Payment:  Credit Card
Address:  123 Main St, Quezon City
------------------------------
ITEMS PURCHASED:
  1. Headphones [Electronics]
     Qty: 1 | Price: ₱1500 | Subtotal: ₱1500
  2. USB-C Cable [Accessories]
     Qty: 2 | Price: ₱250 | Subtotal: ₱500
------------------------------
GRAND TOTAL: ₱2000

[ ORDER ID: 2 ]
Customer: Bob White
Date:     2025-08-03
Status:   Pending
Payment:  Cash on Delivery
Address:  45 Mabini St, Davao City
------------------------------
ITEMS PURCHASED:
  1. Smartphone [Electronics]
     Qty: 1 | Price: ₱22000 | Subtotal: ₱22000
  2. Phone Case [Accessories]
     Qty: 1 | Price: ₱500 | Subtotal: ₱500
------------------------------
GRAND TOTAL: ₱22500

[ ORDER ID: 3 ]
Customer: Charlie Green
Date:     